In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
import numpy as np
import struct
from pathlib import Path
from tqdm import tqdm
import os
import time
import pandas as pd

from torch.profiler import profile, ProfilerActivity, record_function

USE_AMP = True

import tensorflow as tf
import numpy as np




2026-01-24 16:44:32.935513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769273073.131069      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769273073.187013      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769273073.646554      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769273073.646598      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769273073.646601      55 computation_placer.cc:177] computation placer alr

In [2]:
def load_idx_images(path):
    with open(path, 'rb') as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        images = np.frombuffer(f.read(), dtype=np.uint8)
        images = images.reshape(num, rows, cols)
    return images

def load_idx_labels(path):
    with open(path, 'rb') as f:
        magic, num = struct.unpack(">II", f.read(8))
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels


In [3]:
class MNIST_IDX(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]

        img = np.expand_dims(img, axis=2)  # (H, W, 1)

        if self.transform:
            img = self.transform(img)

        return img, label


In [4]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


In [5]:
# train_images = load_idx_images("train-images.idx3-ubyte")
# train_labels = load_idx_labels("train-labels.idx1-ubyte")

# test_images = load_idx_images("t10k-images.idx3-ubyte")
# test_labels = load_idx_labels("t10k-labels.idx1-ubyte")

(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

full_train_dataset = MNIST_IDX(train_images, train_labels, transform)
test_dataset = MNIST_IDX(test_images, test_labels, transform)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
train_size = int(0.875 * len(full_train_dataset))  # 42k
val_size = len(full_train_dataset) - train_size    # 6k

train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size]
)


In [7]:
def get_model(model_name):
    if model_name == "resnet18":
        model = models.resnet18(pretrained=False)
    elif model_name == "resnet50":
        model = models.resnet50(pretrained=False)
    else:
        raise ValueError("Invalid model")

    model.fc = nn.Linear(model.fc.in_features, 10)
    return model


In [8]:
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler(enabled=USE_AMP)

def train_one_epoch(model, loader, optimizer, criterion, device, epoch, epochs):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc=f"Epoch [{epoch}/{epochs}]",
        leave=False
    )

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    return running_loss / len(loader)


/tmp/ipykernel_55/3522788096.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)


In [10]:
# def train_one_epoch(model, loader, optimizer, criterion, device, epoch, epochs):
#     model.train()
#     running_loss = 0.0

#     progress_bar = tqdm(
#         loader,
#         desc=f"Epoch [{epoch}/{epochs}]",
#         leave=False
#     )

#     for images, labels in progress_bar:
#         images, labels = images.to(device), labels.to(device)

#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()
#         progress_bar.set_postfix(loss=loss.item())

#     return running_loss / len(loader)


In [9]:
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total


In [12]:


# def run_experiment(
#     model_name,
#     batch_size,
#     optimizer_name,
#     lr,
#     epochs=5,
#     dataset_name = "MNIST"
# ):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print("====================================")
#     print(f"Dataset: {dataset_name}")
#     print(f"Model: {model_name}")
#     print(f"Device: {device}")
#     print("====================================")

#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size)
#     test_loader = DataLoader(test_dataset, batch_size=batch_size)

#     model = get_model(model_name).to(device)
#     criterion = nn.CrossEntropyLoss()

#     if optimizer_name == "sgd":
#         optimizer = optim.SGD(
#             model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4
#         )
#     elif optimizer_name == "adam":
#         optimizer = optim.Adam(model.parameters(), lr=lr)
#     else:
#         raise ValueError("Invalid optimizer")

#     # -----------------------------
#     # Torch Profiler setup
#     # -----------------------------
#     activities = [ProfilerActivity.CPU]
#     if device.type == "cuda":
#         activities.append(ProfilerActivity.CUDA)

#     prof = profile(
#         activities=activities,
#         record_shapes=True,
#         profile_memory=True,
#         with_flops=True,
#     )

#     total_runtime_ms = 0.0
#     total_flops = 0

#     for epoch in range(epochs):
#         if epoch == 0:
#             prof.start()

#         with record_function("train_epoch"):
#             train_loss = train_one_epoch(
#                 model,
#                 train_loader,
#                 optimizer,
#                 criterion,
#                 device,
#                 epoch + 1,
#                 epochs,
#             )

#         if epoch == 0:
#             prof.stop()

#             events = prof.key_averages()

#             cpu_time_ms = sum(
#                 evt.self_cpu_time_total for evt in events
#             ) / 1000.0  # µs → ms

#             cuda_time_ms = 0.0
#             if device.type == "cuda":
#                 cuda_time_ms = sum(
#                     evt.self_cuda_time_total for evt in events
#                 ) / 1000.0

#             total_runtime_ms = cpu_time_ms + cuda_time_ms

#             total_flops = sum(
#                 evt.flops for evt in events if evt.flops is not None
#             )

#         val_acc = evaluate(model, val_loader, device)
#         print(
#             f"Epoch [{epoch+1}/{epochs}] | "
#             f"Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}%"
#         )

#     test_acc = evaluate(model, test_loader, device)

#     print("\n===== Final Results =====")
#     print(f"Dataset: {dataset_name}")
#     print(f"Test Accuracy: {test_acc:.2f}%")
#     print(f"Runtime (Epoch 1): {total_runtime_ms:.2f} ms")
#     print(f"Total FLOPs (Epoch 1): {total_flops / 1e9:.3f} GFLOPs")

#     result = {
#         "Dataset": dataset_name,
#         "Model": model_name,
#         "Batch Size": batch_size,
#         "Optimizer": optimizer_name.upper(),
#         "Learning Rate": lr,
#         "Epochs": epochs,
#         "Test Accuracy (%)": round(test_acc, 2),
#         "Runtime (ms)": round(total_runtime_ms, 2),
#         "Total FLOPs": total_flops,
#         "Total FLOPs (GFLOPs)": round(total_flops / 1e9, 3),
#         "Device": str(device),
#     }

#     pd.DataFrame([result]).to_csv(
#     "experiment_results.csv",
#     mode="a",
#     header=not os.path.exists("experiment_results.csv"),
#     index=False,
#     )

#     return result


In [13]:
# def run_experiment(model_name, batch_size, optimizer_name, lr, epochs=5, dataset_name = "MNIST"):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print("====================================")
#     print(f"Dataset: {dataset_name}")
#     print(f"Model: {model_name}")
#     print(f"Device: {device}")
#     print("====================================")

#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size)
#     test_loader = DataLoader(test_dataset, batch_size=batch_size)

#     model = get_model(model_name).to(device)
#     criterion = nn.CrossEntropyLoss()

#     if optimizer_name == "sgd":
#         optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
#     elif optimizer_name == "adam":
#         optimizer = optim.Adam(model.parameters(), lr=lr)
#     else:
#         raise ValueError("Invalid optimizer")

#     total_runtime_ms = time.now()
#     for epoch in range(epochs):
#         train_loss = train_loss = train_one_epoch(model, train_loader, optimizer, criterion,device, epoch + 1, epochs)
#         val_acc = evaluate(model, val_loader, device)
#         print(f"Epoch [{epoch+1}/{epochs}] | Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}%")

#     test_acc = evaluate(model, test_loader, device)
#     print("\n===== Final Results =====")
#     print(f"Dataset: {dataset_name}")
#     print(f"Test Accuracy: {test_acc:.2f}%")
#     print(f"Runtime (Epoch 1): {total_runtime_ms:.2f} ms")
#     print(f"Total FLOPs (Epoch 1): {total_flops / 1e9:.3f} GFLOPs")

#     result = {
#         "Dataset": dataset_name,
#         "Model": model_name,
#         "Batch Size": batch_size,
#         "Optimizer": optimizer_name.upper(),
#         "Learning Rate": lr,
#         "Epochs": epochs,
#         "Test Accuracy (%)": round(test_acc, 2),
#         "Runtime (ms)": round(total_runtime_ms, 2),
#         "Device": str(device),
#     }

#     pd.DataFrame([result]).to_csv(
#     "experiment_results.csv",
#     mode="a",
#     header=not os.path.exists("experiment_results.csv"),
#     index=False,
#     )

#     return result


In [19]:


def run_experiment(model_name, batch_size, optimizer_name, lr, epochs=5, dataset_name="MNIST"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("====================================")
    print(f"Dataset: {dataset_name}")
    print(f"Model: {model_name}")
    print(f"Device: {device}")
    print("====================================")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    model = get_model(model_name).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name.lower() == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    elif optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError("Invalid optimizer")

    epoch_times_ms = []
    start_total = time.perf_counter()

    for epoch in range(epochs):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start_epoch = time.perf_counter()

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch + 1, epochs)
        val_acc = evaluate(model, val_loader, device)

        if device.type == 'cuda':
            torch.cuda.synchronize()
        end_epoch = time.perf_counter()

        epoch_time_ms = (end_epoch - start_epoch) * 1000  # convert to ms
        epoch_times_ms.append(epoch_time_ms)

        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}% | Epoch Time: {epoch_time_ms:.2f} ms")

    if device.type == 'cuda':
        torch.cuda.synchronize()
    end_total = time.perf_counter()
    total_runtime_ms = (end_total - start_total) * 1000

    test_acc = evaluate(model, test_loader, device)

    print("\n===== Final Results =====")
    print(f"Dataset: {dataset_name}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Epoch 1 Runtime: {epoch_times_ms[0]:.2f} ms")
    print(f"Total Runtime: {total_runtime_ms:.2f} ms")

    result = {
        "Dataset": dataset_name,
        "Model": model_name,
        "Batch Size": batch_size,
        "Optimizer": optimizer_name.upper(),
        "Learning Rate": lr,
        "Epochs": epochs,
        "Test Accuracy (%)": round(test_acc, 2),
        "Epoch 1 Runtime (ms)": round(epoch_times_ms[0], 2),
        "Total Runtime (ms)": round(total_runtime_ms, 2),
        "Device": str(device),
    }

    pd.DataFrame([result]).to_csv(
        "experiment_results.csv",
        mode="a",
        header=not os.path.exists("experiment_results.csv"),
        index=False,
    )

    return result


In [20]:
run_experiment(
    model_name="resnet18",
    batch_size=16,
    optimizer_name="adam",
    lr=0.001,
    epochs=5
)

Dataset: MNIST
Model: resnet18
Device: cuda


Epoch [1/5]:   0%|          | 0/3282 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 0.1266 | Val Acc: 98.37% | Epoch Time: 194081.62 ms


Epoch [2/5] | Loss: 0.0563 | Val Acc: 98.73% | Epoch Time: 194491.80 ms


Epoch [3/5] | Loss: 0.0428 | Val Acc: 99.16% | Epoch Time: 194225.41 ms


Epoch [4/5] | Loss: 0.0344 | Val Acc: 99.43% | Epoch Time: 194573.23 ms


Epoch [5/5] | Loss: 0.0243 | Val Acc: 99.33% | Epoch Time: 194173.58 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 99.28%
Epoch 1 Runtime: 194081.62 ms
Total Runtime: 971547.00 ms


{'Dataset': 'MNIST',
 'Model': 'resnet18',
 'Batch Size': 16,
 'Optimizer': 'ADAM',
 'Learning Rate': 0.001,
 'Epochs': 5,
 'Test Accuracy (%)': 99.28,
 'Epoch 1 Runtime (ms)': 194081.62,
 'Total Runtime (ms)': 971547.0,
 'Device': 'cuda'}

In [21]:
run_experiment(
    model_name="resnet18",
    batch_size=16,
    optimizer_name="adam",
    lr=0.0001,
    epochs=5
)


Dataset: MNIST
Model: resnet18
Device: cuda


Epoch [1/5]:   0%|          | 0/3282 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 0.1230 | Val Acc: 99.20% | Epoch Time: 191809.11 ms


Epoch [2/5] | Loss: 0.0467 | Val Acc: 98.88% | Epoch Time: 191901.01 ms


Epoch [3/5] | Loss: 0.0341 | Val Acc: 99.31% | Epoch Time: 191594.99 ms


Epoch [4/5] | Loss: 0.0251 | Val Acc: 99.33% | Epoch Time: 191723.14 ms


Epoch [5/5] | Loss: 0.0218 | Val Acc: 99.11% | Epoch Time: 191641.39 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 99.21%
Epoch 1 Runtime: 191809.11 ms
Total Runtime: 958670.95 ms


{'Dataset': 'MNIST',
 'Model': 'resnet18',
 'Batch Size': 16,
 'Optimizer': 'ADAM',
 'Learning Rate': 0.0001,
 'Epochs': 5,
 'Test Accuracy (%)': 99.21,
 'Epoch 1 Runtime (ms)': 191809.11,
 'Total Runtime (ms)': 958670.95,
 'Device': 'cuda'}

In [22]:
run_experiment(
    model_name="resnet18",
    batch_size=32,
    optimizer_name="adam",
    lr=0.001,
    epochs=5
)


Dataset: MNIST
Model: resnet18
Device: cuda


Epoch [1/5]:   0%|          | 0/1641 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 0.1132 | Val Acc: 98.67% | Epoch Time: 173681.16 ms


Epoch [2/5] | Loss: 0.0483 | Val Acc: 91.08% | Epoch Time: 172579.26 ms


Epoch [3/5] | Loss: 0.0396 | Val Acc: 98.55% | Epoch Time: 172262.91 ms


Epoch [4/5] | Loss: 0.0298 | Val Acc: 99.23% | Epoch Time: 171412.29 ms


Epoch [5/5] | Loss: 0.0270 | Val Acc: 99.19% | Epoch Time: 169533.84 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 99.00%
Epoch 1 Runtime: 173681.16 ms
Total Runtime: 859470.50 ms


{'Dataset': 'MNIST',
 'Model': 'resnet18',
 'Batch Size': 32,
 'Optimizer': 'ADAM',
 'Learning Rate': 0.001,
 'Epochs': 5,
 'Test Accuracy (%)': 99.0,
 'Epoch 1 Runtime (ms)': 173681.16,
 'Total Runtime (ms)': 859470.5,
 'Device': 'cuda'}

In [23]:
run_experiment(
    model_name="resnet18",
    batch_size=32,
    optimizer_name="adam",
    lr=0.0001,
    epochs=5
)

Dataset: MNIST
Model: resnet18
Device: cuda


Epoch [1/5]:   0%|          | 0/1641 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 0.1325 | Val Acc: 98.55% | Epoch Time: 171601.70 ms


Epoch [2/5] | Loss: 0.0387 | Val Acc: 96.43% | Epoch Time: 171562.79 ms


Epoch [3/5] | Loss: 0.0290 | Val Acc: 99.37% | Epoch Time: 171522.13 ms


Epoch [4/5] | Loss: 0.0232 | Val Acc: 99.23% | Epoch Time: 172287.62 ms


Epoch [5/5] | Loss: 0.0187 | Val Acc: 99.23% | Epoch Time: 172347.83 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 99.27%
Epoch 1 Runtime: 171601.70 ms
Total Runtime: 859323.41 ms


{'Dataset': 'MNIST',
 'Model': 'resnet18',
 'Batch Size': 32,
 'Optimizer': 'ADAM',
 'Learning Rate': 0.0001,
 'Epochs': 5,
 'Test Accuracy (%)': 99.27,
 'Epoch 1 Runtime (ms)': 171601.7,
 'Total Runtime (ms)': 859323.41,
 'Device': 'cuda'}

In [24]:
run_experiment(
    model_name="resnet18",
    batch_size=16,
    optimizer_name="sgd",
    lr=0.001,
    epochs=5
)

Dataset: MNIST
Model: resnet18
Device: cuda


Epoch [1/5]:   0%|          | 0/3282 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 0.3044 | Val Acc: 98.29% | Epoch Time: 188122.99 ms


Epoch [2/5] | Loss: 0.0574 | Val Acc: 99.09% | Epoch Time: 188399.17 ms


Epoch [3/5] | Loss: 0.0384 | Val Acc: 98.81% | Epoch Time: 188480.05 ms


Epoch [4/5] | Loss: 0.0288 | Val Acc: 99.27% | Epoch Time: 188669.38 ms


Epoch [5/5] | Loss: 0.0226 | Val Acc: 99.31% | Epoch Time: 188306.84 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 99.33%
Epoch 1 Runtime: 188122.99 ms
Total Runtime: 941979.42 ms


{'Dataset': 'MNIST',
 'Model': 'resnet18',
 'Batch Size': 16,
 'Optimizer': 'SGD',
 'Learning Rate': 0.001,
 'Epochs': 5,
 'Test Accuracy (%)': 99.33,
 'Epoch 1 Runtime (ms)': 188122.99,
 'Total Runtime (ms)': 941979.42,
 'Device': 'cuda'}

In [25]:
run_experiment(
    model_name="resnet18",
    batch_size=16,
    optimizer_name="sgd",
    lr=0.0001,
    epochs=5
)


Dataset: MNIST
Model: resnet18
Device: cuda


Epoch [1/5]:   0%|          | 0/3282 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 1.2877 | Val Acc: 92.35% | Epoch Time: 188738.58 ms


Epoch [2/5] | Loss: 0.3126 | Val Acc: 96.27% | Epoch Time: 188457.76 ms


Epoch [3/5] | Loss: 0.1740 | Val Acc: 97.24% | Epoch Time: 188786.91 ms


Epoch [4/5] | Loss: 0.1275 | Val Acc: 97.60% | Epoch Time: 188645.62 ms


Epoch [5/5] | Loss: 0.1042 | Val Acc: 97.89% | Epoch Time: 188867.75 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 97.90%
Epoch 1 Runtime: 188738.58 ms
Total Runtime: 943497.58 ms


{'Dataset': 'MNIST',
 'Model': 'resnet18',
 'Batch Size': 16,
 'Optimizer': 'SGD',
 'Learning Rate': 0.0001,
 'Epochs': 5,
 'Test Accuracy (%)': 97.9,
 'Epoch 1 Runtime (ms)': 188738.58,
 'Total Runtime (ms)': 943497.58,
 'Device': 'cuda'}

In [26]:
run_experiment(
    model_name="resnet18",
    batch_size=32,
    optimizer_name="sgd",
    lr=0.001,
    epochs=5
)

Dataset: MNIST
Model: resnet18
Device: cuda


Epoch [1/5]:   0%|          | 0/1641 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 0.4313 | Val Acc: 97.59% | Epoch Time: 171433.19 ms


Epoch [2/5] | Loss: 0.0704 | Val Acc: 98.53% | Epoch Time: 171258.78 ms


Epoch [3/5] | Loss: 0.0463 | Val Acc: 98.80% | Epoch Time: 170030.10 ms


Epoch [4/5] | Loss: 0.0331 | Val Acc: 99.04% | Epoch Time: 172239.79 ms


Epoch [5/5] | Loss: 0.0260 | Val Acc: 99.16% | Epoch Time: 171974.49 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 99.27%
Epoch 1 Runtime: 171433.19 ms
Total Runtime: 856937.30 ms


{'Dataset': 'MNIST',
 'Model': 'resnet18',
 'Batch Size': 32,
 'Optimizer': 'SGD',
 'Learning Rate': 0.001,
 'Epochs': 5,
 'Test Accuracy (%)': 99.27,
 'Epoch 1 Runtime (ms)': 171433.19,
 'Total Runtime (ms)': 856937.3,
 'Device': 'cuda'}

In [27]:
run_experiment(
    model_name="resnet18",
    batch_size=32,
    optimizer_name="sgd",
    lr=0.0001,
    epochs=5
)

Dataset: MNIST
Model: resnet18
Device: cuda


Epoch [1/5]:   0%|          | 0/1641 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 1.6314 | Val Acc: 79.21% | Epoch Time: 171568.77 ms


Epoch [2/5] | Loss: 0.6830 | Val Acc: 93.37% | Epoch Time: 171639.20 ms


Epoch [3/5] | Loss: 0.3092 | Val Acc: 95.00% | Epoch Time: 171916.03 ms


Epoch [4/5] | Loss: 0.2030 | Val Acc: 96.20% | Epoch Time: 171760.08 ms


Epoch [5/5] | Loss: 0.1564 | Val Acc: 96.95% | Epoch Time: 170850.48 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 97.25%
Epoch 1 Runtime: 171568.77 ms
Total Runtime: 857735.55 ms


{'Dataset': 'MNIST',
 'Model': 'resnet18',
 'Batch Size': 32,
 'Optimizer': 'SGD',
 'Learning Rate': 0.0001,
 'Epochs': 5,
 'Test Accuracy (%)': 97.25,
 'Epoch 1 Runtime (ms)': 171568.77,
 'Total Runtime (ms)': 857735.55,
 'Device': 'cuda'}

In [28]:
run_experiment(
    model_name="resnet50",
    batch_size=16,
    optimizer_name="adam",
    lr=0.001,
    epochs=5
)


Dataset: MNIST
Model: resnet50
Device: cuda


Epoch [1/5]:   0%|          | 0/3282 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


KeyboardInterrupt: 

In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=16,
    optimizer_name="adam",
    lr=0.0001,
    epochs=5
)


In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=32,
    optimizer_name="adam",
    lr=0.001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=32,
    optimizer_name="adam",
    lr=0.0001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=16,
    optimizer_name="sgd",
    lr=0.001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=16,
    optimizer_name="sgd",
    lr=0.0001,
    epochs=5
)

In [30]:
run_experiment(
    model_name="resnet50",
    batch_size=32,
    optimizer_name="sgd",
    lr=0.001,
    epochs=5
)



Dataset: MNIST
Model: resnet50
Device: cuda


Epoch [1/5]:   0%|          | 0/1641 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 0.5682 | Val Acc: 97.35% | Epoch Time: 332183.83 ms


Epoch [2/5] | Loss: 0.0846 | Val Acc: 98.45% | Epoch Time: 331962.84 ms


Epoch [3/5] | Loss: 0.0476 | Val Acc: 98.79% | Epoch Time: 331174.64 ms


Epoch [4/5] | Loss: 0.0305 | Val Acc: 98.83% | Epoch Time: 330808.31 ms


Epoch [5/5] | Loss: 0.0198 | Val Acc: 99.08% | Epoch Time: 329857.54 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 99.12%
Epoch 1 Runtime: 332183.83 ms
Total Runtime: 1655987.91 ms


{'Dataset': 'MNIST',
 'Model': 'resnet50',
 'Batch Size': 32,
 'Optimizer': 'SGD',
 'Learning Rate': 0.001,
 'Epochs': 5,
 'Test Accuracy (%)': 99.12,
 'Epoch 1 Runtime (ms)': 332183.83,
 'Total Runtime (ms)': 1655987.91,
 'Device': 'cuda'}

In [31]:
run_experiment(
    model_name="resnet50",
    batch_size=32,
    optimizer_name="sgd",
    lr=0.0001,
    epochs=5
)



Dataset: MNIST
Model: resnet50
Device: cuda


Epoch [1/5]:   0%|          | 0/1641 [00:00<?, ?it/s]/tmp/ipykernel_55/3522788096.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=USE_AMP):


Epoch [1/5] | Loss: 1.8384 | Val Acc: 45.40% | Epoch Time: 331837.15 ms


Epoch [2/5] | Loss: 1.2404 | Val Acc: 76.25% | Epoch Time: 331942.36 ms


Epoch [3/5] | Loss: 0.6460 | Val Acc: 89.71% | Epoch Time: 331401.79 ms


Epoch [4/5] | Loss: 0.3551 | Val Acc: 94.23% | Epoch Time: 331380.94 ms


Epoch [5/5] | Loss: 0.2314 | Val Acc: 95.55% | Epoch Time: 331510.30 ms

===== Final Results =====
Dataset: MNIST
Test Accuracy: 95.78%
Epoch 1 Runtime: 331837.15 ms
Total Runtime: 1658073.58 ms


{'Dataset': 'MNIST',
 'Model': 'resnet50',
 'Batch Size': 32,
 'Optimizer': 'SGD',
 'Learning Rate': 0.0001,
 'Epochs': 5,
 'Test Accuracy (%)': 95.78,
 'Epoch 1 Runtime (ms)': 331837.15,
 'Total Runtime (ms)': 1658073.58,
 'Device': 'cuda'}

In [ ]:
import tensorflow as tf


# Load FashionMNIST from TensorFlow
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Convert to numpy (safety)
train_images = x_train.astype("uint8")
train_labels = y_train.astype("uint8")
test_images = x_test.astype("uint8")
test_labels = y_test.astype("uint8")

In [ ]:
full_train_dataset = MNIST_IDX(train_images, train_labels, transform)
test_dataset = MNIST_IDX(test_images, test_labels, transform)


In [ ]:
train_size = int(0.875 * len(full_train_dataset))  # 42k
val_size = len(full_train_dataset) - train_size    # 6k

train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size]
)

In [ ]:


# def run_experiment(
#     model_name,
#     batch_size,
#     optimizer_name,
#     lr,
#     epochs=5,
#     dataset_name = "FashionMNIST"
# ):
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print("====================================")
#     print(f"Dataset: {dataset_name}")
#     print(f"Model: {model_name}")
#     print(f"Device: {device}")
#     print("====================================")

#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size)
#     test_loader = DataLoader(test_dataset, batch_size=batch_size)

#     model = get_model(model_name).to(device)
#     criterion = nn.CrossEntropyLoss()

#     if optimizer_name == "sgd":
#         optimizer = optim.SGD(
#             model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4
#         )
#     elif optimizer_name == "adam":
#         optimizer = optim.Adam(model.parameters(), lr=lr)
#     else:
#         raise ValueError("Invalid optimizer")

#     # -----------------------------
#     # Torch Profiler setup
#     # -----------------------------
#     activities = [ProfilerActivity.CPU]
#     if device.type == "cuda":
#         activities.append(ProfilerActivity.CUDA)

#     prof = profile(
#         activities=activities,
#         record_shapes=True,
#         profile_memory=True,
#         with_flops=True,
#     )

#     total_runtime_ms = 0.0
#     total_flops = 0

#     for epoch in range(epochs):
#         if epoch == 0:
#             prof.start()

#         with record_function("train_epoch"):
#             train_loss = train_one_epoch(
#                 model,
#                 train_loader,
#                 optimizer,
#                 criterion,
#                 device,
#                 epoch + 1,
#                 epochs,
#             )

#         if epoch == 0:
#             prof.stop()

#             events = prof.key_averages()

#             cpu_time_ms = sum(
#                 evt.self_cpu_time_total for evt in events
#             ) / 1000.0  # µs → ms

#             cuda_time_ms = 0.0
#             if device.type == "cuda":
#                 cuda_time_ms = sum(
#                     evt.self_cuda_time_total for evt in events
#                 ) / 1000.0

#             total_runtime_ms = cpu_time_ms + cuda_time_ms

#             total_flops = sum(
#                 evt.flops for evt in events if evt.flops is not None
#             )

#         val_acc = evaluate(model, val_loader, device)
#         print(
#             f"Epoch [{epoch+1}/{epochs}] | "
#             f"Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}%"
#         )

#     test_acc = evaluate(model, test_loader, device)

#     print("\n===== Final Results =====")
#     print(f"Dataset: {dataset_name}")
#     print(f"Test Accuracy: {test_acc:.2f}%")
#     print(f"Runtime (Epoch 1): {total_runtime_ms:.2f} ms")
#     print(f"Total FLOPs (Epoch 1): {total_flops / 1e9:.3f} GFLOPs")

#     result = {
#         "Dataset": dataset_name,
#         "Model": model_name,
#         "Batch Size": batch_size,
#         "Optimizer": optimizer_name.upper(),
#         "Learning Rate": lr,
#         "Epochs": epochs,
#         "Test Accuracy (%)": round(test_acc, 2),
#         "Runtime (ms)": round(total_runtime_ms, 2),
#         "Total FLOPs": total_flops,
#         "Total FLOPs (GFLOPs)": round(total_flops / 1e9, 3),
#         "Device": str(device),
#     }

#     pd.DataFrame([result]).to_csv(
#     "experiment_results.csv",
#     mode="a",
#     header=not os.path.exists("experiment_results.csv"),
#     index=False,
#     )

#     return result


In [ ]:


def run_experiment(model_name, batch_size, optimizer_name, lr, epochs=5, dataset_name="FashionMNIST"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("====================================")
    print(f"Dataset: {dataset_name}")
    print(f"Model: {model_name}")
    print(f"Device: {device}")
    print("====================================")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    model = get_model(model_name).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name.lower() == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    elif optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError("Invalid optimizer")

    epoch_times_ms = []
    start_total = time.perf_counter()

    for epoch in range(epochs):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start_epoch = time.perf_counter()

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, epoch + 1, epochs)
        val_acc = evaluate(model, val_loader, device)

        if device.type == 'cuda':
            torch.cuda.synchronize()
        end_epoch = time.perf_counter()

        epoch_time_ms = (end_epoch - start_epoch) * 1000  # convert to ms
        epoch_times_ms.append(epoch_time_ms)

        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}% | Epoch Time: {epoch_time_ms:.2f} ms")

    if device.type == 'cuda':
        torch.cuda.synchronize()
    end_total = time.perf_counter()
    total_runtime_ms = (end_total - start_total) * 1000

    test_acc = evaluate(model, test_loader, device)

    print("\n===== Final Results =====")
    print(f"Dataset: {dataset_name}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Epoch 1 Runtime: {epoch_times_ms[0]:.2f} ms")
    print(f"Total Runtime: {total_runtime_ms:.2f} ms")

    result = {
        "Dataset": dataset_name,
        "Model": model_name,
        "Batch Size": batch_size,
        "Optimizer": optimizer_name.upper(),
        "Learning Rate": lr,
        "Epochs": epochs,
        "Test Accuracy (%)": round(test_acc, 2),
        "Epoch 1 Runtime (ms)": round(epoch_times_ms[0], 2),
        "Total Runtime (ms)": round(total_runtime_ms, 2),
        "Device": str(device),
    }

    pd.DataFrame([result]).to_csv(
        "experiment_results.csv",
        mode="a",
        header=not os.path.exists("experiment_results.csv"),
        index=False,
    )

    return result


In [ ]:
run_experiment(
    model_name="resnet18",
    batch_size=16,
    optimizer_name="adam",
    lr=0.001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet18",
    batch_size=16,
    optimizer_name="adam",
    lr=0.0001,
    epochs=5
)


In [ ]:
run_experiment(
    model_name="resnet18",
    batch_size=32,
    optimizer_name="adam",
    lr=0.001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet18",
    batch_size=32,
    optimizer_name="adam",
    lr=0.0001,
    epochs=5
)



In [ ]:
run_experiment(
    model_name="resnet18",
    batch_size=16,
    optimizer_name="sgd",
    lr=0.001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet18",
    batch_size=16,
    optimizer_name="sgd",
    lr=0.0001,
    epochs=5
)



In [ ]:
run_experiment(
    model_name="resnet18",
    batch_size=32,
    optimizer_name="sgd",
    lr=0.001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet18",
    batch_size=32,
    optimizer_name="sgd",
    lr=0.0001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=16,
    optimizer_name="adam",
    lr=0.001,
    epochs=5
)


In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=16,
    optimizer_name="adam",
    lr=0.0001,
    epochs=5
)



In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=32,
    optimizer_name="adam",
    lr=0.001,
    epochs=5
)


In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=32,
    optimizer_name="adam",
    lr=0.0001,
    epochs=5
)



In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=16,
    optimizer_name="sgd",
    lr=0.001,
    epochs=5
)



In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=16,
    optimizer_name="sgd",
    lr=0.0001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=32,
    optimizer_name="sgd",
    lr=0.001,
    epochs=5
)

In [ ]:
run_experiment(
    model_name="resnet50",
    batch_size=32,
    optimizer_name="sgd",
    lr=0.0001,
    epochs=5
)



In [ ]:
import time
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


In [ ]:
def prepare_svm_data(images, labels, max_samples=None):
    X = images.reshape(images.shape[0], -1) / 255.0
    y = labels

    if max_samples is not None:
        X = X[:max_samples]
        y = y[:max_samples]

    return X, y


In [ ]:
def run_svm(kernel, C, gamma, X_train, y_train, X_test, y_test):
    svm = SVC(kernel=kernel, C=C, gamma=gamma)

    start = time.time()
    svm.fit(X_train, y_train)
    end = time.time()

    train_time_ms = (end - start) * 1000

    y_pred = svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred) * 100

    return acc, train_time_ms


In [ ]:
(x_train_m, y_train_m), (x_test_m, y_test_m) = tf.keras.datasets.mnist.load_data()


In [ ]:
X_train_m, y_train_m = prepare_svm_data(
    x_train_m, y_train_m, max_samples=10000
)

X_test_m, y_test_m = prepare_svm_data(
    x_test_m, y_test_m
)


In [ ]:
(x_train_f, y_train_f), (x_test_f, y_test_f) = tf.keras.datasets.fashion_mnist.load_data()


In [ ]:
X_train_f, y_train_f = prepare_svm_data(
    x_train_f, y_train_f, max_samples=10000
)

X_test_f, y_test_f = prepare_svm_data(
    x_test_f, y_test_f
)


In [ ]:
# MNIST
acc, time_ms = run_svm(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    X_train=X_train_m,
    y_train=y_train_m,
    X_test=X_test_m,
    y_test=y_test_m
)

print(f"MNIST | Accuracy: {acc:.2f}% | Time: {time_ms:.2f} ms")

pd.DataFrame([{"dataset": "MNIST", "kernel": "rbf", "Accuracy": f"{acc:.2f}%", "Time": f"{time_ms:.2f} ms"}]).to_csv(
    "svm_results.csv",
    mode="a",
    header=not os.path.exists("svm_results.csv"),
    index=False,
)


# FashionMNIST
acc, time_ms = run_svm(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    X_train=X_train_f,
    y_train=y_train_f,
    X_test=X_test_f,
    y_test=y_test_f
)

print(f"FashionMNIST | Accuracy: {acc:.2f}% | Time: {time_ms:.2f} ms")

pd.DataFrame([{"dataset": "FashionMNIST","kernel": "rbf", "Accuracy": f"{acc:.2f}%", "Time": f"{time_ms:.2f} ms"}]).to_csv(
    "svm_results.csv",
    mode="a",
    header=not os.path.exists("svm_results.csv"),
    index=False,
)


In [ ]:
# MNIST
acc_poly, time_ms_poly = run_svm(
    kernel="poly",
    C=1.0,
    gamma="scale",
    X_train=X_train_m,
    y_train=y_train_m,
    X_test=X_test_m,
    y_test=y_test_m
)
print(f"MNIST | Accuracy: {acc:.2f}% | Time: {time_ms:.2f} ms")

pd.DataFrame([{"dataset": "MNIST", "kernel": "poly", "Accuracy": f"{acc:.2f}%", "Time": f"{time_ms:.2f} ms"}]).to_csv(
    "svm_results.csv",
    mode="a",
    header=not os.path.exists("svm_results.csv"),
    index=False,
)

# FashionMNIST
acc, time_ms = run_svm(
    kernel="poly",
    C=1.0,
    gamma="scale",
    X_train=X_train_f,
    y_train=y_train_f,
    X_test=X_test_f,
    y_test=y_test_f
)

print(f"FashionMNIST | Accuracy: {acc:.2f}% | Time: {time_ms:.2f} ms")

pd.DataFrame([{"dataset": "FashionMNIST","kernel": "poly", "Accuracy": f"{acc:.2f}%", "Time": f"{time_ms:.2f} ms"}]).to_csv(
    "svm_results.csv",
    mode="a",
    header=not os.path.exists("svm_results.csv"),
    index=False,
)



FLOPs caculation

In [10]:
from torch.profiler import profile, ProfilerActivity
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader
import torch
import torch.nn as nn


def run_flops_experiment(
    model_name,
    batch_size,
    epochs=5,
    dataset_name="MNIST",
    use_amp=False,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("====================================")
    print(f"FLOPs Estimation Only")
    print(f"Dataset: {dataset_name}")
    print(f"Model: {model_name}")
    print(f"Batch Size: {batch_size}")
    print(f"Epochs: {epochs}")
    print(f"Device: {device}")
    print("====================================")

    # Data (only to get ONE batch)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    # Model
    model = get_model(model_name).to(device)
    model.train()  # important: enables backward graph

    criterion = nn.CrossEntropyLoss()

    # ---- Get exactly ONE batch ----
    images, labels = next(iter(train_loader))
    images = images.to(device)
    labels = labels.to(device)

    # Warm-up (recommended)
    with autocast(enabled=use_amp):
        loss = criterion(model(images), labels)
    loss.backward()
    torch.cuda.synchronize() if device.type == "cuda" else None
    model.zero_grad(set_to_none=True)

    # ---- Profile ONE batch ----
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        with_flops=True,
        record_shapes=True,
    ) as prof:
        with autocast(enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        loss.backward()

    # ---- Extract FLOPs ----
    flops_per_batch = sum(
        evt.flops for evt in prof.key_averages()
        if evt.flops is not None
    )

    # ---- Interpolate ----
    total_batches = len(train_loader)
    flops_per_epoch = flops_per_batch * total_batches
    total_flops = flops_per_epoch * epochs

    # ---- Report ----
    print("\n========== FLOPs ESTIMATE ==========")
    print(f"Device         : {str(device)}")
    print(f"Dataset Name   : {dataset_name}")
    print(f"Model Name     : {model_name}")
    print(f"FLOPs per batch : {flops_per_batch:.3e}")
    print(f"Total batches  : {total_batches}")
    print(f"Epochs         : {epochs}")
    print(f"FLOPs / epoch  : {flops_per_epoch:.3e}")
    print(f"Total FLOPs    : {total_flops:.3e}")
    print("===================================\n")

    # return {
    #     "Dataset": dataset_name,
    #     "Model": model_name,
    #     "Batch Size": batch_size,
    #     "Epochs": epochs,
    #     "Device": str(device),
    #     "FLOPs per Batch": flops_per_batch,
    #     "FLOPs per Epoch": flops_per_epoch,
    #     "Total FLOPs": total_flops,
    # }


In [13]:
run_flops_experiment(
    model_name="resnet18",
    batch_size=16,
    epochs=5
)

FLOPs Estimation Only
Dataset: MNIST
Model: resnet18
Batch Size: 16
Epochs: 5
Device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipykernel_55/4280120449.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):



========== FLOPs ESTIMATE ==========
Device         : cuda
Dataset Name   : MNIST
Model Name     : resnet18
FLOPs per batch : 5.803e+10
Total batches  : 3282
Epochs         : 5
FLOPs / epoch  : 1.905e+14
Total FLOPs    : 9.523e+14



/tmp/ipykernel_55/4280120449.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


In [14]:
run_flops_experiment(
    model_name="resnet18",
    batch_size=32,
    epochs=5
)

FLOPs Estimation Only
Dataset: MNIST
Model: resnet18
Batch Size: 32
Epochs: 5
Device: cuda


/tmp/ipykernel_55/4280120449.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_55/4280120449.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):



========== FLOPs ESTIMATE ==========
Device         : cuda
Dataset Name   : MNIST
Model Name     : resnet18
FLOPs per batch : 1.161e+11
Total batches  : 1641
Epochs         : 5
FLOPs / epoch  : 1.905e+14
Total FLOPs    : 9.523e+14



In [15]:
run_flops_experiment(
    model_name="resnet50",
    batch_size=16,
    epochs=5
)

FLOPs Estimation Only
Dataset: MNIST
Model: resnet50
Batch Size: 16
Epochs: 5
Device: cuda


/tmp/ipykernel_55/4280120449.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_55/4280120449.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):



========== FLOPs ESTIMATE ==========
Device         : cuda
Dataset Name   : MNIST
Model Name     : resnet50
FLOPs per batch : 1.308e+11
Total batches  : 3282
Epochs         : 5
FLOPs / epoch  : 4.293e+14
Total FLOPs    : 2.146e+15



In [16]:
run_flops_experiment(
    model_name="resnet50",
    batch_size=32,
    epochs=5
)

FLOPs Estimation Only
Dataset: MNIST
Model: resnet50
Batch Size: 32
Epochs: 5
Device: cuda


/tmp/ipykernel_55/4280120449.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_55/4280120449.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):



========== FLOPs ESTIMATE ==========
Device         : cuda
Dataset Name   : MNIST
Model Name     : resnet50
FLOPs per batch : 2.616e+11
Total batches  : 1641
Epochs         : 5
FLOPs / epoch  : 4.293e+14
Total FLOPs    : 2.146e+15

